In [1]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, T5ForConditionalGeneration, T5Tokenizer
import torch
import pandas as pd
import json
pd.set_option('display.max_columns', None) 
pd.set_option('max_colwidth', None) # show full width of showing cols
pd.set_option("expand_frame_repr", False) # print cols side by side as it's supposed to be

In [2]:
with open("Model_dataset/cv.json", "r") as file:
    CV_DATA= json.load(file)

In [3]:
q_type_models= [
    'model/fine_tuned_question_classifier_model_lite-default',
    'model/fine_tuned_question_classifier_model_lite-Adam',
    'model/fine_tuned_question_classifier_model_lite-AdamW',
    'model/fine_tuned_question_classifier_model_lite-SGD']
qa_type_model= 't5-large'

# Comparing results for all different optimiser result:

In [4]:
def get_dataframe_for_comparision(questions):
    table_data= {
        "questions": questions
    }
    for q_type_model in q_type_models:
        QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_model)
        QUESTION_CLASSIFIER_MODEL.eval()
        QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_model)
        ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label

        def get_question_type_predictions(texts):
            inputs = QUESTION_CLASSIFIER_TOKENIZER(texts, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
            logits = outputs.logits
            predicted_classes = torch.argmax(logits, dim=1)
            id2label = QUESTION_CLASSIFIER_MODEL.config.id2label
            return [id2label[idx.item()] for idx in predicted_classes]
        table_data[q_type_model.split("-")[-1]]= get_question_type_predictions(questions)
    return pd.DataFrame(table_data)

In [5]:
questions = [
    "What have you used for web development?",
    "How do I train a neural network?",
    "How much do you want to earn?",
    "What are you looking for in the new company?",
    "How much break do you need per day?",
    "Share you most recent salary draw.",
    "Rate yourself 1 to 10 for python developer.",
    "What is your most recent degree?",
    "Have you completed masters?",
    "Have you completed Bsc",
    "Who is the CEO of your company?",
    "Who inspire you the most?",
    "have you rechived any performance bounus?",
    "How frequent you expect performance review?",
    "Have you review others code before?"
    "What are the top framework you use?",
    "Have you completed any personal project recently?",
    "Have you a lead a team before?",
    "How many years of experience do you have with SQL?",
    "What you expect from the current company?",
    "How many years of work experience do you have with Microsoft Products?",
    "How many years of work experience do you have with Microsoft Fabric?",
    "Have you completed the following level of education: Bachelor's Degree?",
    "Mobile phone number",
    "Phone country code",
    "Email address",
    "How many years of work experience do you have with Python (Programming Language)?",
    "How many years of work experience do you have with Google BigQuery?",
    "How many years of work experience do you have with Terraform?",
]
get_dataframe_for_comparision(questions)

,questions,default,Adam,AdamW,SGD
0,What have you used for web development?,working_experince,working_experince,working_experince,working_experince
1,How do I train a neural network?,working_experince,working_experince,working_experince,working_experince
2,How much do you want to earn?,expected_ctc,expected_ctc,expected_ctc,expected_ctc
3,What are you looking for in the new company?,availability,availability,personal_information,expected_ctc
4,How much break do you need per day?,expected_ctc,expected_ctc,availability,expected_ctc
5,Share you most recent salary draw.,current_ctc,current_ctc,current_ctc,current_ctc
6,Rate yourself 1 to 10 for python developer.,skills,skills,skills,skills
7,What is your most recent degree?,education,education,education,education
8,Have you completed masters?,education,education,education,working_experince
9,Have you completed Bsc,education,education,education,education


# Using classifier

In [6]:
QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_models[0])
QUESTION_CLASSIFIER_MODEL.eval()
QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_models[0])
ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label

In [7]:
def get_question_type_prediction(text):
    inputs = QUESTION_CLASSIFIER_TOKENIZER(text, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
    logits = outputs.logits
    predicted_classes = torch.argmax(logits, dim=1)
    id2label = QUESTION_CLASSIFIER_MODEL.config.id2label
    return id2label[predicted_classes.item()]
# get_question_type_prediction("What have you used for web development?")

In [8]:
def get_question_type_predictions(texts):
    inputs = QUESTION_CLASSIFIER_TOKENIZER(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
    logits = outputs.logits
    predicted_classes = torch.argmax(logits, dim=1)
    id2label = QUESTION_CLASSIFIER_MODEL.config.id2label
    return [id2label[idx.item()] for idx in predicted_classes]

# Example usage
questions = [
    "What have you used for web development?",
    "How do I train a neural network?",
    "Who is the CEO of OpenAI?",
    "How much do you want to earn?",
    "What are you looking for in the new company?",
    "How much break do you need per day?",
    "Share you most recent salary draw.",
    "Rate yourself 1 to 10 for python developer.",
    "What is your most recent degree?",
    "Have you completed masters?",
    "Have you completed Bsc",
    "Who is the CEO of your company?",
    "Who inspire you the most?",
    "have you rechived any performance bounus?",
    "How frequent you expect performance review?"
]
# get_question_type_predictions(questions)

# Question answer model

In [9]:
# QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_type_model)
# QUESTION_ANSWER_MODEL.eval()
# QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_type_model, legacy= False)

In [10]:
# def get_model_out_raw(question):
#     predicted_question_type= get_question_type_prediction(question)
#     context= CV_DATA[predicted_question_type]
#     input_text = f"question: {question} context: {context}"
#     inputs = QUESTION_ANSWER_TOKENIZER(input_text, return_tensors="pt")
#     outputs = QUESTION_ANSWER_MODEL.generate(input_ids=inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True)

#     return predicted_question_type, QUESTION_ANSWER_TOKENIZER.decode(outputs[0], skip_special_tokens=True)


# # get_model_out_raw("Total work experince in python?")